# 📁 Notebook 1: The Proxy Problem

Why routing large files through your servers is a terrible idea.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why large files don't belong in databases
- The cost of proxying data through servers
- When to use blob storage vs databases

In [ ]:
import time
import os

print("✅ Ready to learn about large blob handling!")

## 💾 Why Blob Storage?

In [ ]:
print("💾 Databases vs Blob Storage")
print("=" * 60)
print("""
DATABASES are great for:
─────────────────────────────────────────────────────────────
✅ Structured data with relationships
✅ Complex queries (JOINs, aggregations)
✅ Transactions and ACID guarantees
✅ Frequent updates to small records

DATABASES are terrible for:
─────────────────────────────────────────────────────────────
❌ Large binary objects (videos, images)
❌ Files that don't need SQL queries
❌ Data > 10MB per record

WHY?
• 100MB BLOB kills query performance
• Backups take forever
• Replication slows down
• Storage costs 10-100x more
""")

In [ ]:
print("📊 Storage Cost Comparison")
print("=" * 60)

storage_costs = {
    "PostgreSQL RDS": 0.115,
    "MySQL RDS": 0.115,
    "S3 Standard": 0.023,
    "S3 Infrequent": 0.0125,
    "S3 Glacier": 0.004
}

file_size_gb = 100

print(f"\n💰 Cost to store {file_size_gb}GB of files (per month):")
print()
for storage, cost_per_gb in storage_costs.items():
    monthly_cost = file_size_gb * cost_per_gb
    bar = "$" * int(monthly_cost / 0.5)
    print(f"   {storage:20}: ${monthly_cost:>6.2f} {bar}")

print("\n✅ Blob storage is 5-30x cheaper!")

## 🔄 The Proxy Anti-Pattern

In [ ]:
print("🔄 The Proxy Anti-Pattern")
print("=" * 60)
print("""
Traditional approach: Route everything through your servers

UPLOAD:
─────────────────────────────────────────────────────────────
┌────────┐    2GB     ┌────────────┐    2GB     ┌─────────┐
│ Client │ ─────────> │ API Server │ ─────────> │   S3    │
└────────┘            └────────────┘            └─────────┘
                           │
                      Holds 2GB in
                      memory/disk!

DOWNLOAD:
─────────────────────────────────────────────────────────────
┌────────┐    2GB     ┌────────────┐    2GB     ┌─────────┐
│ Client │ <───────── │ API Server │ <───────── │   S3    │
└────────┘            └────────────┘            └─────────┘

PROBLEMS:
• Server memory/disk exhausted
• Network bandwidth doubled (in + out)
• Latency added for no reason
• Server can't handle other requests
• You PAY for egress twice!
""")

In [ ]:
print("⏱️ Time to Upload (Back-of-Envelope)")
print("=" * 60)

file_size_gb = 2
file_size_bits = file_size_gb * 8 * 1024 * 1024 * 1024

scenarios = {
    "Home WiFi (100 Mbps)": 100_000_000,
    "Mobile 4G (50 Mbps)": 50_000_000,
    "Office (1 Gbps)": 1_000_000_000,
}

print(f"\n📤 Uploading {file_size_gb}GB video:")
print()
for scenario, bandwidth_bps in scenarios.items():
    time_seconds = file_size_bits / bandwidth_bps
    minutes = int(time_seconds // 60)
    seconds = int(time_seconds % 60)
    print(f"   {scenario:25}: {minutes}m {seconds}s")

print("\n❓ What if connection drops at 99%?")
print("   Without resumable uploads: START OVER! 😱")

In [ ]:
print("💸 Bandwidth Cost Analysis")
print("=" * 60)

uploads_per_day = 10000
avg_file_size_gb = 0.5
days_per_month = 30

total_gb_per_month = uploads_per_day * avg_file_size_gb * days_per_month

egress_cost_per_gb = 0.09

print(f"\n📊 Scenario: Video platform")
print(f"   Uploads per day: {uploads_per_day:,}")
print(f"   Average file size: {avg_file_size_gb}GB")
print(f"   Monthly data: {total_gb_per_month:,.0f}GB")

print(f"\n💰 PROXY approach (data flows through server):")
proxy_cost = total_gb_per_month * egress_cost_per_gb * 2
print(f"   Egress cost: ${proxy_cost:,.2f}/month")
print(f"   (Paying twice: client→server + server→S3)")

print(f"\n💰 DIRECT approach (presigned URLs):")
direct_cost = 0
print(f"   Egress cost: ${direct_cost:.2f}/month")
print(f"   (Upload directly to S3, no egress!)")

print(f"\n✅ Savings: ${proxy_cost:,.2f}/month!")

## ✅ The Solution Preview

In [ ]:
print("✅ The Better Way: Direct Upload")
print("=" * 60)
print("""
Instead of proxying, give clients temporary credentials.

STEP 1: Client asks for permission
─────────────────────────────────────────────────────────────
┌────────┐  "I want to    ┌────────────┐
│ Client │ ─────────────> │ API Server │
│        │  upload 2GB"   │            │
└────────┘                └─────┬──────┘
                                │
                          Validates user,
                          checks quota,
                          generates URL

STEP 2: Server returns presigned URL
─────────────────────────────────────────────────────────────
┌────────┐  "Here's your  ┌────────────┐
│ Client │ <───────────── │ API Server │
│        │  upload URL"   │            │
└────────┘                └────────────┘

STEP 3: Client uploads directly to storage
─────────────────────────────────────────────────────────────
┌────────┐       2GB      ┌─────────┐
│ Client │ ─────────────> │   S3    │
│        │  (direct!)     │         │
└────────┘                └─────────┘

Your server never touches the 2GB!
""")

## 🧪 Quick Quiz

1. **Why shouldn't you store a 100MB video in PostgreSQL?**

2. **What's wrong with proxying uploads through your API server?**

3. **When IS it okay to proxy data through your server?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why not store 100MB video in PostgreSQL:")
print("   - Kills query performance")
print("   - Slows backups and replication")
print("   - 5-30x more expensive than S3")
print("   - Can't do SQL queries on video bytes anyway")
print()
print("2. Problems with proxying:")
print("   - Uses server memory/bandwidth")
print("   - Adds latency for no value")
print("   - Pay for egress twice")
print("   - Blocks server from other work")
print()
print("3. When proxying IS okay:")
print("   - Small files (< 10MB)")
print("   - Need real-time validation")
print("   - Compliance requires inspection")
print("   - User needs immediate feedback")

## 📚 Summary

### Key Takeaways

1. **Databases for metadata** - Structured data, queries, relationships
2. **Blob storage for files** - Cheap, scalable, durable
3. **Proxying is expensive** - Memory, bandwidth, latency, cost
4. **Direct upload is better** - Client → Storage directly
5. **Rule of thumb** - If > 10MB and no SQL queries needed, use blob storage

### Next Up

In **Notebook 2**, we'll implement presigned URLs:
- Generate upload/download URLs
- Security constraints
- Using MinIO (S3-compatible)